In [ ]:
# Cell 1 - Install dependencies (~2 minutes)
!pip install --target=/kaggle/working -U vllm pyngrok

In [ ]:
# Cell 2 - Set ngrok token
from pyngrok import ngrok
ngrok.set_auth_token("REPLACE_WITH_YOUR_NGROK_TOKEN")

In [ ]:
# Cell 3 - Fix libcuda.so for FlashInfer
import subprocess, os, re, pathlib

# Step 1: Find actual libcuda.so.1 location
candidates = [
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    "/usr/local/cuda/lib64/libcuda.so.1",
    "/usr/local/cuda/lib64/libcuda.so",
]
libcuda_real = None
for c in candidates:
    if os.path.exists(c):
        libcuda_real = c
        break

if not libcuda_real:
    r = subprocess.run("find / -name 'libcuda.so*' 2>/dev/null | head -5",
                       shell=True, capture_output=True, text=True)
    for line in r.stdout.strip().splitlines():
        if os.path.exists(line):
            libcuda_real = line
            break

print(f"libcuda.so.1 found at: {libcuda_real}")

# Step 2: Create correct symlinks in /tmp
for p in ["/tmp/libcuda.so", "/tmp/cuda_stubs/libcuda.so"]:
    if os.path.lexists(p): os.remove(p)
os.makedirs("/tmp/cuda_stubs", exist_ok=True)
os.symlink(libcuda_real, "/tmp/libcuda.so")
os.symlink(libcuda_real, "/tmp/cuda_stubs/libcuda.so")
print(f"Symlinks created: /tmp/libcuda.so -> {libcuda_real}")

# Step 3: Try mount --bind (makes stubs dir contain libcuda.so at expected path)
subprocess.run("cp /usr/local/cuda/lib64/stubs/* /tmp/cuda_stubs/ 2>/dev/null || true", shell=True)
r = subprocess.run(["mount", "--bind", "/tmp/cuda_stubs", "/usr/local/cuda/lib64/stubs"],
                   capture_output=True, text=True)
print(f"mount --bind: {'OK' if r.returncode == 0 else r.stderr.strip()}")

# Step 4: Patch FlashInfer cpp_ext.py to add -L/tmp to linker flags
cpp_ext = "/kaggle/working/flashinfer/jit/cpp_ext.py"
src = pathlib.Path(cpp_ext).read_text()
patched = re.sub(r'("-L[^"]*stubs")', r'"-L/tmp", \1', src)
if patched != src:
    pathlib.Path(cpp_ext).write_text(patched)
    print("Patched cpp_ext.py: added -L/tmp")

# Step 5: Clear FlashInfer cache to force recompile with fix
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"])
print("Cleared FlashInfer cache. Ready to run Cell 4.")

In [ ]:
# Cell 4 - Start vLLM server (~60-90 seconds to load model)
import subprocess, threading, time, os

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"/tmp:{env.get('LD_LIBRARY_PATH', '')}"

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--enforce-eager"
    ], env=env)

threading.Thread(target=run_vllm, daemon=True).start()
time.sleep(90)
print("vLLM started!")

In [ ]:
# Cell 5 - Create ngrok tunnel -> COPY the URL printed below into .env
tunnel = ngrok.connect(8001, "http")
print("="*50)
print("COPY DONG NAY VAO FILE .env TREN MAY LOCAL:")
print(f"VLLM_NGROK_URL={tunnel.public_url}")
print("="*50)

In [ ]:
# Cell 6 - Embedding service (optional)
!pip install -q sentence-transformers fastapi uvicorn

from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

embed_app = FastAPI()
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@embed_app.post("/embed")
def embed(data: dict):
    return {"embeddings": embed_model.encode(data["texts"]).tolist()}

threading.Thread(target=lambda: uvicorn.run(embed_app, host="0.0.0.0", port=8002), daemon=True).start()

embed_tunnel = ngrok.connect(8002, "http")
print("="*50)
print(f"EMBED_NGROK_URL={embed_tunnel.public_url}")
print("="*50)

In [ ]:
# Cell 7 - Keep session alive
import time
print("Session is alive. Keep this cell running...")
while True:
    time.sleep(300)
    print("Still alive...")